# Práctica N.º 05 — Listas Enlazadas Simples, Dobles y Circulares

**Curso:** Algoritmos y Estructuras de Datos — SIS210  
**Semana:** 5  
**Docente:** Dr. Aldo Hernán Zanabria Gálvez  
**Estudiante:** _[Apellido Nombre]_  
**Fecha:** _[dd/mm/aaaa]_  

---

Este notebook desarrolla las actividades en **Python 3.11+** de la guía de práctica:

- **Actividad 1:** Lista enlazada simple completa.
- **Actividad 2:** Pruebas unitarias con `unittest`.
- **Actividad 4:** Lista doblemente enlazada y eliminación O(1) dado el nodo.
- **Actividad 5:** Algoritmo de Floyd (detectar y localizar el inicio de un ciclo).

## Índice

1. [Celda de imports y utilidades comunes](#imports)
2. [Actividad 1 — Lista enlazada simple completa](#actividad-1)
3. [Actividad 2 — Pruebas unitarias con `unittest`](#actividad-2)
4. [Actividad 4 — Lista doblemente enlazada y eliminación O(1)](#actividad-4)
5. [Actividad 5 — Algoritmo de Floyd: detección y localización de ciclos](#actividad-5)

---

### Resumen de complejidades objetivo

| Operación | Lista simple | Lista doble |
|-----------|--------------|-------------|
| Insertar al inicio | O(1) | O(1) |
| Insertar al final | O(1) con puntero `cola` | O(1) con puntero `cola` |
| Insertar en posición k | O(k) | O(k) |
| Eliminar por dato | O(n) | O(n) |
| Eliminar nodo dado (referencia) | O(n) | **O(1)** |
| Buscar | O(n) | O(n) |
| Invertir | O(n) | O(n) |
| Detección de ciclo (Floyd) | O(n) tiempo / O(1) espacio | — |

In [2]:
# ============================================================
# Celda 3 — Imports y utilidades comunes
# ============================================================
# Todo lo que necesitamos para las Actividades 1, 2, 4 y 5.
# ============================================================

from __future__ import annotations

from dataclasses import dataclass
from typing import Optional, Iterator, Any
import unittest
import sys

# ------------------------------------------------------------
# Utilidad: tamaño aproximado en memoria de un objeto Python
# (la usaremos más adelante para comparar listas vs. vectores)
# ------------------------------------------------------------
def tamano_bytes(obj: Any) -> int:
    """Devuelve el tamaño en bytes de un objeto según sys.getsizeof."""
    return sys.getsizeof(obj)


# ------------------------------------------------------------
# Pequeña ayuda visual para imprimir resultados en el notebook
# ------------------------------------------------------------
def titulo(texto: str) -> None:
    """Imprime un título formateado para separar secciones."""
    print("\n" + "=" * 60)
    print(texto)
    print("=" * 60)


print("Entorno listo. Python", sys.version.split()[0])

Entorno listo. Python 3.14.6


<a id="actividad-1"></a>
## Actividad 1 — Python: Lista enlazada simple completa

**Objetivo:** implementar la clase `ListaEnlazada` con todas las operaciones requeridas, usando `dataclass` para el nodo y manteniendo un puntero extra `cola` que permite inserción al final en **O(1)**.

### Operaciones a implementar

| Método | Descripción | Complejidad |
|--------|-------------|-------------|
| `insertar_inicio(dato)` | Inserta un nodo al frente | O(1) |
| `insertar_final(dato)` | Inserta un nodo al final usando `cola` | O(1) |
| `insertar_en_posicion(pos, dato)` | Inserta en la posición indicada | O(k) |
| `eliminar(dato)` | Elimina el primer nodo con ese dato; retorna `bool` | O(n) |
| `buscar(dato)` | Retorna el nodo o `None` | O(n) |
| `invertir()` | Invierte la lista in-place con 3 punteros | O(n) |
| `__len__` | Retorna el tamaño | O(1) |
| `__iter__` | Recorre los datos | O(n) |
| `__repr__` | Representación `a -> b -> NULL` | O(n) |

### Casos borde a considerar

- Insertar en lista vacía (actualizar `cabeza` **y** `cola`).
- Insertar en posición `0` → delega en `insertar_inicio`.
- Insertar en posición `len` → delega en `insertar_final`.
- Eliminar la cabeza cuando hay un solo nodo (dejar `cabeza = cola = None`).
- Eliminar la cola (actualizar `cola` al nodo anterior).
- Invertir una lista vacía o de un solo nodo.

In [3]:
# ============================================================
# Celda 5 — Actividad 1: Nodo + ListaEnlazada
# ============================================================

@dataclass
class Nodo:
    """Nodo de una lista enlazada simple."""
    dato: Any
    siguiente: Optional["Nodo"] = None


class ListaEnlazada:
    """Lista enlazada simple con puntero extra 'cola' para O(1) al final."""

    # --------------------------------------------------------
    # Constructor
    # --------------------------------------------------------
    def __init__(self) -> None:
        self.cabeza: Optional[Nodo] = None
        self.cola: Optional[Nodo] = None
        self._tamanio: int = 0

    # --------------------------------------------------------
    # Inserción
    # --------------------------------------------------------
    def insertar_inicio(self, dato: Any) -> None:
        nuevo = Nodo(dato)
        if self.cabeza is None:
            # Lista vacía: cabeza y cola apuntan al único nodo
            self.cabeza = self.cola = nuevo
        else:
            nuevo.siguiente = self.cabeza
            self.cabeza = nuevo
        self._tamanio += 1

    def insertar_final(self, dato: Any) -> None:
        nuevo = Nodo(dato)
        if self.cola is None:
            # Lista vacía
            self.cabeza = self.cola = nuevo
        else:
            # Usamos el puntero 'cola' → O(1)
            self.cola.siguiente = nuevo
            self.cola = nuevo
        self._tamanio += 1

    def insertar_en_posicion(self, posicion: int, dato: Any) -> None:
        if posicion < 0 or posicion > self._tamanio:
            raise IndexError(f"Posición fuera de rango: {posicion}")

        # Casos rápidos
        if posicion == 0:
            self.insertar_inicio(dato)
            return
        if posicion == self._tamanio:
            self.insertar_final(dato)
            return

        # Caso general: recorrer hasta el nodo anterior a 'posicion'
        anterior = self.cabeza
        for _ in range(posicion - 1):
            anterior = anterior.siguiente  # type: ignore[union-attr]

        nuevo = Nodo(dato)
        nuevo.siguiente = anterior.siguiente  # type: ignore[union-attr]
        anterior.siguiente = nuevo            # type: ignore[union-attr]
        self._tamanio += 1

    # --------------------------------------------------------
    # Eliminación
    # --------------------------------------------------------
    def eliminar(self, dato: Any) -> bool:
        actual = self.cabeza
        anterior: Optional[Nodo] = None

        while actual is not None:
            if actual.dato == dato:
                # --- Caso 1: eliminar la cabeza ---
                if anterior is None:
                    self.cabeza = actual.siguiente
                    # Si la lista quedó vacía, también cola = None
                    if self.cabeza is None:
                        self.cola = None
                # --- Caso 2: eliminar un nodo intermedio o la cola ---
                else:
                    anterior.siguiente = actual.siguiente
                    if actual is self.cola:
                        self.cola = anterior
                self._tamanio -= 1
                return True

            anterior = actual
            actual = actual.siguiente

        return False  # no se encontró el dato

    # --------------------------------------------------------
    # Búsqueda
    # --------------------------------------------------------
    def buscar(self, dato: Any) -> Optional[Nodo]:
        actual = self.cabeza
        while actual is not None:
            if actual.dato == dato:
                return actual
            actual = actual.siguiente
        return None

    # --------------------------------------------------------
    # Utilidades
    # --------------------------------------------------------
    def invertir(self) -> None:
        """Invierte la lista in-place con 3 punteros: prev, actual, siguiente."""
        prev: Optional[Nodo] = None
        actual: Optional[Nodo] = self.cabeza
        # La cola pasa a ser la antigua cabeza
        self.cola = self.cabeza

        while actual is not None:
            siguiente = actual.siguiente   # guardar ANTES de modificar
            actual.siguiente = prev        # invertir el enlace
            prev = actual                  # avanzar prev
            actual = siguiente             # avanzar actual

        self.cabeza = prev                 # nueva cabeza

    # --------------------------------------------------------
    # Protocolo de contenedor
    # --------------------------------------------------------
    def __len__(self) -> int:
        return self._tamanio

    def __iter__(self) -> Iterator[Any]:
        actual = self.cabeza
        while actual is not None:
            yield actual.dato
            actual = actual.siguiente

    def __repr__(self) -> str:
        if self.cabeza is None:
            return "NULL"
        return " -> ".join(str(d) for d in self) + " -> NULL"

In [4]:
# ============================================================
# Celda 6 — Actividad 1: pruebas manuales rápidas
# ============================================================

titulo("Actividad 1 — Demostración de ListaEnlazada")

l = ListaEnlazada()

# --- Inserción al final ---
for v in [10, 20, 30, 40, 50]:
    l.insertar_final(v)
print("Lista inicial (insertar_final):     ", l)
print("Tamaño:", len(l), "| cabeza:", l.cabeza.dato, "| cola:", l.cola.dato)

# --- Inserción al inicio ---
l.insertar_inicio(5)
print("\nTras insertar_inicio(5):            ", l)

# --- Inserción en posición media ---
l.insertar_en_posicion(3, 25)   # entre 20 y 30 (recordando que ahora empieza en 5)
print("Tras insertar_en_posicion(3, 25):   ", l)

# --- Búsqueda ---
nodo = l.buscar(30)
print("\nbuscar(30) ->", nodo.dato if nodo else None)
print("buscar(99) ->", l.buscar(99))

# --- Eliminar cabeza ---
l.eliminar(5)
print("\nTras eliminar(5) [cabeza]:          ", l)

# --- Eliminar cola ---
l.eliminar(50)
print("Tras eliminar(50) [cola]:           ", l)

# --- Eliminar intermedio ---
l.eliminar(25)
print("Tras eliminar(25) [intermedio]:     ", l)

# --- Eliminar inexistente ---
print("eliminar(999) ->", l.eliminar(999))
print("Estado actual:                      ", l)

# --- Invertir ---
l.invertir()
print("\nTras invertir():                    ", l)
print("cabeza:", l.cabeza.dato, "| cola:", l.cola.dato)

# --- Lista vacía ---
lv = ListaEnlazada()
print("\nLista vacía:", lv, "| len:", len(lv))
print("eliminar(1) en vacía ->", lv.eliminar(1))


Actividad 1 — Demostración de ListaEnlazada
Lista inicial (insertar_final):      10 -> 20 -> 30 -> 40 -> 50 -> NULL
Tamaño: 5 | cabeza: 10 | cola: 50

Tras insertar_inicio(5):             5 -> 10 -> 20 -> 30 -> 40 -> 50 -> NULL
Tras insertar_en_posicion(3, 25):    5 -> 10 -> 20 -> 25 -> 30 -> 40 -> 50 -> NULL

buscar(30) -> 30
buscar(99) -> None

Tras eliminar(5) [cabeza]:           10 -> 20 -> 25 -> 30 -> 40 -> 50 -> NULL
Tras eliminar(50) [cola]:            10 -> 20 -> 25 -> 30 -> 40 -> NULL
Tras eliminar(25) [intermedio]:      10 -> 20 -> 30 -> 40 -> NULL
eliminar(999) -> False
Estado actual:                       10 -> 20 -> 30 -> 40 -> NULL

Tras invertir():                     40 -> 30 -> 20 -> 10 -> NULL
cabeza: 40 | cola: 10

Lista vacía: NULL | len: 0
eliminar(1) en vacía -> False


In [5]:
# ============================================================
# Celda 6 — Actividad 1: pruebas manuales rápidas + casos borde
# ============================================================

titulo("Actividad 1 — Demostración de ListaEnlazada")

l = ListaEnlazada()

# --- Inserción al final ---
for v in [10, 20, 30, 40, 50]:
    l.insertar_final(v)
print("Lista inicial (insertar_final):     ", l)
print("Tamaño:", len(l), "| cabeza:", l.cabeza.dato, "| cola:", l.cola.dato)

# --- Inserción al inicio ---
l.insertar_inicio(5)
print("\nTras insertar_inicio(5):            ", l)

# --- Inserción en posición media ---
l.insertar_en_posicion(3, 25)
print("Tras insertar_en_posicion(3, 25):   ", l)

# --- Búsqueda ---
nodo = l.buscar(30)
print("\nbuscar(30) ->", nodo.dato if nodo else None)
print("buscar(99) ->", l.buscar(99))

# --- Eliminar cabeza ---
l.eliminar(5)
print("\nTras eliminar(5) [cabeza]:          ", l)

# --- Eliminar cola ---
l.eliminar(50)
print("Tras eliminar(50) [cola]:           ", l)

# --- Eliminar intermedio ---
l.eliminar(25)
print("Tras eliminar(25) [intermedio]:     ", l)

# --- Eliminar inexistente ---
print("eliminar(999) ->", l.eliminar(999))
print("Estado actual:                      ", l)

# --- Invertir ---
l.invertir()
print("\nTras invertir():                    ", l)
print("cabeza:", l.cabeza.dato, "| cola:", l.cola.dato)


# ============================================================
# Casos borde adicionales
# ============================================================
titulo("Casos borde")

# --- 1) Lista vacía ---
lv = ListaEnlazada()
print("Lista vacía:", lv, "| len:", len(lv))
print("eliminar(1) en vacía ->", lv.eliminar(1))
lv.invertir()
print("invertir() en vacía ->", lv, "| cabeza:", lv.cabeza, "| cola:", lv.cola)

# --- 2) Un solo nodo: eliminar deja la lista vacía y cola=None ---
lu = ListaEnlazada()
lu.insertar_final(42)
print("\nUn solo nodo:", lu)
print("eliminar(42) ->", lu.eliminar(42))
print("Después:       ", lu, "| cabeza:", lu.cabeza, "| cola:", lu.cola, "| len:", len(lu))

# --- 3) Invertir lista de un solo nodo ---
lu2 = ListaEnlazada()
lu2.insertar_final(7)
lu2.invertir()
print("\nInvertir 1 nodo:", lu2, "| cabeza:", lu2.cabeza.dato, "| cola:", lu2.cola.dato)

# --- 4) insertar_en_posicion(0, ...) delega en insertar_inicio ---
lp = ListaEnlazada()
for v in [1, 2, 3]:
    lp.insertar_final(v)
lp.insertar_en_posicion(0, 0)
print("\ninsertar_en_posicion(0, 0):", lp)

# --- 5) insertar_en_posicion(len, ...) delega en insertar_final ---
lp.insertar_en_posicion(len(lp), 99)
print("insertar_en_posicion(len, 99):", lp)

# --- 6) Posición fuera de rango ---
try:
    lp.insertar_en_posicion(100, -1)
except IndexError as e:
    print("\nIndexError capturado:", e)


Actividad 1 — Demostración de ListaEnlazada
Lista inicial (insertar_final):      10 -> 20 -> 30 -> 40 -> 50 -> NULL
Tamaño: 5 | cabeza: 10 | cola: 50

Tras insertar_inicio(5):             5 -> 10 -> 20 -> 30 -> 40 -> 50 -> NULL
Tras insertar_en_posicion(3, 25):    5 -> 10 -> 20 -> 25 -> 30 -> 40 -> 50 -> NULL

buscar(30) -> 30
buscar(99) -> None

Tras eliminar(5) [cabeza]:           10 -> 20 -> 25 -> 30 -> 40 -> 50 -> NULL
Tras eliminar(50) [cola]:            10 -> 20 -> 25 -> 30 -> 40 -> NULL
Tras eliminar(25) [intermedio]:      10 -> 20 -> 30 -> 40 -> NULL
eliminar(999) -> False
Estado actual:                       10 -> 20 -> 30 -> 40 -> NULL

Tras invertir():                     40 -> 30 -> 20 -> 10 -> NULL
cabeza: 40 | cola: 10

Casos borde
Lista vacía: NULL | len: 0
eliminar(1) en vacía -> False
invertir() en vacía -> NULL | cabeza: None | cola: None

Un solo nodo: 42 -> NULL
eliminar(42) -> True
Después:        NULL | cabeza: None | cola: None | len: 0

Invertir 1 nodo: 7 -> NU

<a id="actividad-2"></a>
## Actividad 2 — Python: Pruebas unitarias con `unittest`

**Objetivo:** desarrollar un conjunto completo de pruebas unitarias que verifique **todas** las operaciones de la clase `ListaEnlazada`, usando el módulo estándar `unittest`.

### Estrategia de pruebas

- Se usa `setUp()` para crear una **lista fresca antes de cada prueba** con los valores `[10, 20, 30, 40, 50]`.
- Cada prueba es independiente: el estado no se comparte entre métodos.
- Se cubren:
  - Inserción al inicio, al final y en posición media.
  - Eliminación de cabeza, cola y nodo intermedio.
  - Eliminación de un dato inexistente (retorna `False`).
  - Inversión de la lista.
  - Comportamiento de una lista vacía.
- Se ejecuta con `unittest.main(verbosity=2)` para ver cada prueba individual.

### Pruebas incluidas

| # | Prueba | Qué verifica |
|---|--------|--------------|
| 1 | `test_insertar_inicio_actualiza_cabeza` | Nueva cabeza y tamaño |
| 2 | `test_insertar_final_usa_puntero_cola` | Nuevo último elemento |
| 3 | `test_insertar_en_posicion_media` | Orden correcto tras insertar |
| 4 | `test_eliminar_cabeza` | Cabeza y tamaño actualizados |
| 5 | `test_eliminar_cola` | Cola y tamaño actualizados |
| 6 | `test_eliminar_medio` | Orden correcto tras eliminar |
| 7 | `test_eliminar_no_existente_retorna_false` | Retorno `False` y lista intacta |
| 8 | `test_invertir` | Lista invertida correctamente |
| 9 | `test_lista_vacia` | `len == 0`, `cabeza is None`, eliminar retorna `False` |
| 10 | `test_insertar_en_posicion_invalida` | Lanza `IndexError` |
| 11 | `test_eliminar_unico_nodo_deja_lista_vacia` | `cabeza` y `cola` quedan en `None` |
| 12 | `test_invertir_lista_un_nodo` | No rompe con un solo nodo |

> **Nota:** en el notebook, `unittest.main()` no funciona directamente porque Jupyter ya tiene su propio bucle de ejecución. Por eso usaremos `unittest.TextTestRunner` con `verbosity=2` sobre una `TestSuite`, y así veremos cada prueba con su nombre y resultado.

In [6]:
# ============================================================
# Celda 8 — Actividad 2: Pruebas unitarias con unittest
# ============================================================

class TestListaEnlazada(unittest.TestCase):
    """Conjunto de pruebas unitarias para ListaEnlazada."""

    def setUp(self) -> None:
        """Se ejecuta antes de CADA prueba: crea una lista fresca."""
        self.l = ListaEnlazada()
        for v in [10, 20, 30, 40, 50]:
            self.l.insertar_final(v)

    # --------------------------------------------------------
    # Inserción
    # --------------------------------------------------------
    def test_insertar_inicio_actualiza_cabeza(self) -> None:
        self.l.insertar_inicio(5)
        self.assertEqual(list(self.l)[0], 5)
        self.assertEqual(len(self.l), 6)

    def test_insertar_final_usa_puntero_cola(self) -> None:
        self.l.insertar_final(60)
        self.assertEqual(list(self.l)[-1], 60)
        self.assertEqual(self.l.cola.dato, 60)
        self.assertEqual(len(self.l), 6)

    def test_insertar_en_posicion_media(self) -> None:
        self.l.insertar_en_posicion(2, 25)
        self.assertEqual(list(self.l), [10, 20, 25, 30, 40, 50])
        self.assertEqual(len(self.l), 6)

    def test_insertar_en_posicion_invalida(self) -> None:
        with self.assertRaises(IndexError):
            self.l.insertar_en_posicion(100, -1)
        with self.assertRaises(IndexError):
            self.l.insertar_en_posicion(-1, -1)

    # --------------------------------------------------------
    # Eliminación
    # --------------------------------------------------------
    def test_eliminar_cabeza(self) -> None:
        self.assertTrue(self.l.eliminar(10))
        self.assertEqual(list(self.l)[0], 20)
        self.assertEqual(len(self.l), 4)

    def test_eliminar_cola(self) -> None:
        self.assertTrue(self.l.eliminar(50))
        self.assertEqual(list(self.l)[-1], 40)
        self.assertEqual(self.l.cola.dato, 40)
        self.assertEqual(len(self.l), 4)

    def test_eliminar_medio(self) -> None:
        self.assertTrue(self.l.eliminar(30))
        self.assertEqual(list(self.l), [10, 20, 40, 50])
        self.assertEqual(len(self.l), 4)

    def test_eliminar_no_existente_retorna_false(self) -> None:
        resultado = self.l.eliminar(99)
        self.assertFalse(resultado)
        self.assertEqual(len(self.l), 5)
        self.assertEqual(list(self.l), [10, 20, 30, 40, 50])

    def test_eliminar_unico_nodo_deja_lista_vacia(self) -> None:
        lu = ListaEnlazada()
        lu.insertar_final(42)
        self.assertTrue(lu.eliminar(42))
        self.assertIsNone(lu.cabeza)
        self.assertIsNone(lu.cola)
        self.assertEqual(len(lu), 0)

    # --------------------------------------------------------
    # Inversión
    # --------------------------------------------------------
    def test_invertir(self) -> None:
        self.l.invertir()
        self.assertEqual(list(self.l), [50, 40, 30, 20, 10])
        self.assertEqual(self.l.cabeza.dato, 50)
        self.assertEqual(self.l.cola.dato, 10)

    def test_invertir_lista_un_nodo(self) -> None:
        lu = ListaEnlazada()
        lu.insertar_final(7)
        lu.invertir()
        self.assertEqual(list(lu), [7])
        self.assertEqual(lu.cabeza.dato, 7)
        self.assertEqual(lu.cola.dato, 7)

    # --------------------------------------------------------
    # Lista vacía
    # --------------------------------------------------------
    def test_lista_vacia(self) -> None:
        l_vacia = ListaEnlazada()
        self.assertEqual(len(l_vacia), 0)
        self.assertIsNone(l_vacia.cabeza)
        self.assertIsNone(l_vacia.cola)
        self.assertFalse(l_vacia.eliminar(1))
        self.assertIsNone(l_vacia.buscar(1))


# ------------------------------------------------------------
# Ejecución de las pruebas en Jupyter
# (unittest.main() no funciona bien en notebooks)
# ------------------------------------------------------------
def ejecutar_pruebas() -> unittest.result.TestResult:
    loader = unittest.TestLoader()
    suite = loader.loadTestsFromTestCase(TestListaEnlazada)
    runner = unittest.TextTestRunner(verbosity=2)
    return runner.run(suite)


titulo("Actividad 2 — Ejecutando pruebas unitarias")
resultado = ejecutar_pruebas()
print(f"\nPruebas ejecutadas: {resultado.testsRun}")
print(f"Fallos: {len(resultado.failures)} | Errores: {len(resultado.errors)}")

test_eliminar_cabeza (__main__.TestListaEnlazada.test_eliminar_cabeza) ... ok
test_eliminar_cola (__main__.TestListaEnlazada.test_eliminar_cola) ... ok
test_eliminar_medio (__main__.TestListaEnlazada.test_eliminar_medio) ... ok
test_eliminar_no_existente_retorna_false (__main__.TestListaEnlazada.test_eliminar_no_existente_retorna_false) ... ok
test_eliminar_unico_nodo_deja_lista_vacia (__main__.TestListaEnlazada.test_eliminar_unico_nodo_deja_lista_vacia) ... ok
test_insertar_en_posicion_invalida (__main__.TestListaEnlazada.test_insertar_en_posicion_invalida) ... ok
test_insertar_en_posicion_media (__main__.TestListaEnlazada.test_insertar_en_posicion_media) ... ok
test_insertar_final_usa_puntero_cola (__main__.TestListaEnlazada.test_insertar_final_usa_puntero_cola) ... ok
test_insertar_inicio_actualiza_cabeza (__main__.TestListaEnlazada.test_insertar_inicio_actualiza_cabeza) ... ok
test_invertir (__main__.TestListaEnlazada.test_invertir) ... ok
test_invertir_lista_un_nodo (__main__.Test


Actividad 2 — Ejecutando pruebas unitarias

Pruebas ejecutadas: 12
Fallos: 0 | Errores: 0


<a id="actividad-4"></a>
## Actividad 4 — Python: Lista doblemente enlazada y eliminación O(1) dado el nodo

**Objetivo:** implementar una lista doblemente enlazada donde eliminar un nodo dada su **referencia directa** es **O(1)**, sin necesidad de buscar el nodo anterior.

### Diferencia clave frente a la lista simple

| Aspecto | Lista simple | Lista doble |
|---------|--------------|-------------|
| Punteros por nodo | `siguiente` | `siguiente` + `anterior` |
| Eliminar por dato | O(n) | O(n) |
| **Eliminar nodo dado (referencia)** | **O(n)** | **O(1)** |
| Memoria por nodo | dato + 1 puntero | dato + 2 punteros |

### Operaciones a implementar

- `insertar_final(dato) -> NodoDoble` — devuelve el nodo para poder eliminarlo después en O(1).
- `eliminar_nodo(nodo) -> None` — elimina el nodo en O(1) ajustando los enlaces `anterior` y `siguiente`.
- `__len__`, `__iter__`, `__repr__` — para integrarse con Python.

### Casos borde de `eliminar_nodo`

1. Nodo es la **cabeza** → `cabeza = nodo.siguiente`; si existe, `nuevo_cabeza.anterior = None`.
2. Nodo es la **cola** → `cola = nodo.anterior`; si existe, `nuevo_cola.siguiente = None`.
3. Nodo es el **único** → `cabeza = cola = None`.
4. Nodo **intermedio** → enlazar `anterior.siguiente` con `siguiente` y `siguiente.anterior` con `anterior`.

### Aplicación real: LRU Cache simplificado

Un **LRU Cache** (Least Recently Used) combina:
- Una **lista doble** para mantener el orden de uso (más reciente al final).
- Un **diccionario** (hashmap) que mapea clave → nodo.

Cuando se accede a una página, se elimina su nodo en **O(1)** y se reinserta al final en **O(1)**. Esto es la base de `functools.lru_cache` en Python.

In [7]:
# ============================================================
# Celda 10 — Actividad 4: NodoDoble + ListaDoble
# ============================================================

@dataclass
class NodoDoble:
    """Nodo de una lista doblemente enlazada."""
    dato: Any
    siguiente: Optional["NodoDoble"] = None
    anterior: Optional["NodoDoble"] = None


class ListaDoble:
    """Lista doblemente enlazada con eliminación O(1) dado el nodo."""

    # --------------------------------------------------------
    # Constructor
    # --------------------------------------------------------
    def __init__(self) -> None:
        self.cabeza: Optional[NodoDoble] = None
        self.cola: Optional[NodoDoble] = None
        self._n: int = 0

    # --------------------------------------------------------
    # Inserción al final (devuelve el nodo para eliminarlo en O(1))
    # --------------------------------------------------------
    def insertar_final(self, dato: Any) -> NodoDoble:
        nuevo = NodoDoble(dato)
        if self.cola is None:
            # Lista vacía
            self.cabeza = self.cola = nuevo
        else:
            nuevo.anterior = self.cola
            self.cola.siguiente = nuevo
            self.cola = nuevo
        self._n += 1
        return nuevo

    # --------------------------------------------------------
    # Inserción al inicio (por completitud)
    # --------------------------------------------------------
    def insertar_inicio(self, dato: Any) -> NodoDoble:
        nuevo = NodoDoble(dato)
        if self.cabeza is None:
            self.cabeza = self.cola = nuevo
        else:
            nuevo.siguiente = self.cabeza
            self.cabeza.anterior = nuevo
            self.cabeza = nuevo
        self._n += 1
        return nuevo

    # --------------------------------------------------------
    # Eliminación O(1) dado el nodo
    # --------------------------------------------------------
    def eliminar_nodo(self, nodo: NodoDoble) -> None:
        """Elimina UN NODO específico en O(1) — sin buscarlo."""
        if nodo is None:
            return

        # Reajustar el vecino anterior
        if nodo.anterior is not None:
            nodo.anterior.siguiente = nodo.siguiente
        else:
            # Era la cabeza
            self.cabeza = nodo.siguiente

        # Reajustar el vecino siguiente
        if nodo.siguiente is not None:
            nodo.siguiente.anterior = nodo.anterior
        else:
            # Era la cola
            self.cola = nodo.anterior

        # Desconectar el nodo por seguridad
        nodo.anterior = None
        nodo.siguiente = None
        self._n -= 1

    # --------------------------------------------------------
    # Búsqueda (por completitud)
    # --------------------------------------------------------
    def buscar(self, dato: Any) -> Optional[NodoDoble]:
        actual = self.cabeza
        while actual is not None:
            if actual.dato == dato:
                return actual
            actual = actual.siguiente
        return None

    # --------------------------------------------------------
    # Protocolo de contenedor
    # --------------------------------------------------------
    def __len__(self) -> int:
        return self._n

    def __iter__(self) -> Iterator[Any]:
        actual = self.cabeza
        while actual is not None:
            yield actual.dato
            actual = actual.siguiente

    def __repr__(self) -> str:
        if self.cabeza is None:
            return "NULL"
        return " <-> ".join(str(d) for d in self) + " <-> NULL"

In [8]:
# ============================================================
# Celda 11 — Actividad 4: demo del LRU Cache simplificado
#              + pruebas con unittest para ListaDoble
# ============================================================

titulo("Actividad 4 — LRU Cache simplificado")

cache = ListaDoble()
nodo_a = cache.insertar_final("página:A")
nodo_b = cache.insertar_final("página:B")
nodo_c = cache.insertar_final("página:C")

print("Cache inicial:      ", cache)
print("cabeza:", cache.cabeza.dato, "| cola:", cache.cola.dato, "| n:", len(cache))

# 'B' fue accedida recientemente → moverla al final (eliminar y reinsertar)
cache.eliminar_nodo(nodo_b)                       # O(1)
nodo_b = cache.insertar_final("página:B")         # O(1)

print("\nTras acceder a B:   ", cache)
print("cabeza:", cache.cabeza.dato, "| cola:", cache.cola.dato, "| n:", len(cache))

# Acceder a A (la más antigua) → moverla al final
cache.eliminar_nodo(nodo_a)
nodo_a = cache.insertar_final("página:A")
print("\nTras acceder a A:   ", cache)

# Eliminar la cabeza (la menos usada) en O(1)
cache.eliminar_nodo(cache.cabeza)
print("\nTras expulsar LRU:  ", cache)


# ============================================================
# Pruebas unitarias para ListaDoble
# ============================================================
class TestListaDoble(unittest.TestCase):

    def test_insertar_final_devuelve_nodo(self) -> None:
        ld = ListaDoble()
        n1 = ld.insertar_final(1)
        n2 = ld.insertar_final(2)
        self.assertIsInstance(n1, NodoDoble)
        self.assertIs(ld.cabeza, n1)
        self.assertIs(ld.cola, n2)
        self.assertEqual(len(ld), 2)

    def test_eliminar_cabeza(self) -> None:
        ld = ListaDoble()
        n1 = ld.insertar_final(1)
        n2 = ld.insertar_final(2)
        ld.eliminar_nodo(n1)
        self.assertIs(ld.cabeza, n2)
        self.assertIsNone(n2.anterior)
        self.assertEqual(len(ld), 1)

    def test_eliminar_cola(self) -> None:
        ld = ListaDoble()
        n1 = ld.insertar_final(1)
        n2 = ld.insertar_final(2)
        ld.eliminar_nodo(n2)
        self.assertIs(ld.cola, n1)
        self.assertIsNone(n1.siguiente)
        self.assertEqual(len(ld), 1)

    def test_eliminar_intermedio(self) -> None:
        ld = ListaDoble()
        n1 = ld.insertar_final(1)
        n2 = ld.insertar_final(2)
        n3 = ld.insertar_final(3)
        ld.eliminar_nodo(n2)
        self.assertIs(n1.siguiente, n3)
        self.assertIs(n3.anterior, n1)
        self.assertEqual(list(ld), [1, 3])
        self.assertEqual(len(ld), 2)

    def test_eliminar_unico_deja_lista_vacia(self) -> None:
        ld = ListaDoble()
        n1 = ld.insertar_final(1)
        ld.eliminar_nodo(n1)
        self.assertIsNone(ld.cabeza)
        self.assertIsNone(ld.cola)
        self.assertEqual(len(ld), 0)

    def test_mover_al_final_simula_lru(self) -> None:
        ld = ListaDoble()
        a = ld.insertar_final("A")
        b = ld.insertar_final("B")
        c = ld.insertar_final("C")
        # acceder a "A" → moverla al final
        ld.eliminar_nodo(a)
        ld.insertar_final("A")
        self.assertEqual(list(ld), ["B", "C", "A"])


def ejecutar_pruebas_dobles() -> unittest.result.TestResult:
    loader = unittest.TestLoader()
    suite = loader.loadTestsFromTestCase(TestListaDoble)
    runner = unittest.TextTestRunner(verbosity=2)
    return runner.run(suite)


titulo("Actividad 4 — Pruebas unitarias de ListaDoble")
resultado_doble = ejecutar_pruebas_dobles()
print(f"\nPruebas ejecutadas: {resultado_doble.testsRun}")
print(f"Fallos: {len(resultado_doble.failures)} | Errores: {len(resultado_doble.errors)}")

test_eliminar_cabeza (__main__.TestListaDoble.test_eliminar_cabeza) ... ok
test_eliminar_cola (__main__.TestListaDoble.test_eliminar_cola) ... ok
test_eliminar_intermedio (__main__.TestListaDoble.test_eliminar_intermedio) ... ok
test_eliminar_unico_deja_lista_vacia (__main__.TestListaDoble.test_eliminar_unico_deja_lista_vacia) ... ok
test_insertar_final_devuelve_nodo (__main__.TestListaDoble.test_insertar_final_devuelve_nodo) ... ok
test_mover_al_final_simula_lru (__main__.TestListaDoble.test_mover_al_final_simula_lru) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.032s

OK



Actividad 4 — LRU Cache simplificado
Cache inicial:       página:A <-> página:B <-> página:C <-> NULL
cabeza: página:A | cola: página:C | n: 3

Tras acceder a B:    página:A <-> página:C <-> página:B <-> NULL
cabeza: página:A | cola: página:B | n: 3

Tras acceder a A:    página:C <-> página:B <-> página:A <-> NULL

Tras expulsar LRU:   página:B <-> página:A <-> NULL

Actividad 4 — Pruebas unitarias de ListaDoble

Pruebas ejecutadas: 6
Fallos: 0 | Errores: 0


<a id="actividad-5"></a>
## Actividad 5 — Python: Algoritmo de Floyd (detectar y localizar el inicio del ciclo)

**Objetivo:** implementar el algoritmo completo de Floyd (*tortuga y liebre*) que no solo detecta si una lista enlazada tiene un ciclo, sino que además encuentra el **nodo exacto donde comienza** el ciclo, en **O(n) tiempo** y **O(1) espacio**.

---

### ¿Qué es un ciclo en una lista enlazada?

Una lista enlazada simple normalmente termina en `NULL`. Sin embargo, si el puntero `siguiente` de algún nodo apunta hacia atrás a un nodo anterior, se forma un **ciclo**. Esto provoca:

* **Recorrido infinito** si se usa un `while actual is not None` ingenuo.
* **Fuga de memoria** en lenguajes sin recolector de basura.
* **Errores lógicos** difíciles de depurar.

**Lista con ciclo:**
```text
1 → 2 → 3 → 4 → 5
        ↑       │
        └───────┘
```

En el ejemplo anterior, el ciclo comienza en el nodo `3`.

---

### Algoritmo de Floyd — las dos fases

El algoritmo usa dos punteros que recorren la lista a distinta velocidad:

* **`lento`** (tortuga): avanza **1 nodo** por iteración.
* **`rapido`** (liebre): avanza **2 nodos** por iteración.

#### Fase 1 — Detectar si existe un ciclo

```python
lento = rapido = cabeza
mientras rapido y rapido.siguiente:
    lento = lento.siguiente
    rapido = rapido.siguiente.siguiente
    si lento is rapido:
        # ¡colisión! hay ciclo
        romper
si no hubo colisión:
    retornar None # no hay ciclo
```

**¿Por qué funciona?** Si hay ciclo, `rapido` entra primero y da vueltas. En algún momento `lento` también entra, y como `rapido` avanza el doble, la distancia entre ambos se reduce en 1 por iteración hasta que se encuentran.

#### Fase 2 — Encontrar el inicio del ciclo

```python
lento = cabeza
mientras lento is not rapido:
    lento = lento.siguiente
    rapido = rapido.siguiente # ahora ambos avanzan de a 1
retornar lento # ambos apuntan al inicio del ciclo
```

**¿Por qué funciona?** Sea:
* `L` = distancia de la cabeza al inicio del ciclo.
* `C` = longitud del ciclo.
* `k` = distancia del inicio del ciclo al punto de colisión.

En la colisión: `lento` recorrió `L + k`, `rapido` recorrió `L + k + n·C` (para algún entero `n`). Como `rapido` va al doble:

\[2 \cdot (L + k) = L + k + n \cdot C\]
\[\implies L + k = n \cdot C\]
\[\implies L = n \cdot C - k\]

Es decir, `L` es congruente con `−k` módulo `C`. Por eso, si movemos `lento` a la cabeza y ambos avanzan de a 1, se encontrarán exactamente en el **inicio del ciclo**.

---

### Complejidad

| Recurso | Coste |
| :--- | :--- |
| **Tiempo** | \(O(n)\) |
| **Espacio** | \(O(1)\) — solo dos punteros |
| **Modifica la lista** | No |

---

### Comparación con alternativas

| Método | Tiempo | Espacio | Encuentra inicio | Modifica lista |
| :--- | :--- | :--- | :--- | :--- |
| Marcar nodos visitados | \(O(n)\) | \(O(n)\) | Sí | Sí |
| Hash set de nodos | \(O(n)\) | \(O(n)\) | Sí | No |
| **Floyd (tortuga y liebre)** | **\(O(n)\)** | **\(O(1)\)** | **Sí** | **No** |

---

### Casos borde a considerar

* **Lista vacía:** retorna `None` inmediatamente.
* **Un solo nodo sin ciclo:** `rapido.siguiente is None` \(\rightarrow\) retorna `None`.
* **Un solo nodo con ciclo a sí mismo:** colisión inmediata en la Fase 1; en la Fase 2, `lento` vuelve a cabeza y se encuentra con `rapido` en el mismo nodo.
* **Ciclo que empieza en la cabeza:** la Fase 2 retorna la cabeza.
* **Ciclo al final:** se comporta igual que cualquier otro ciclo.

---

### Aplicaciones reales

* **Detección de bucles** en listas enlazadas (problema clásico de entrevistas).
* **Detección de ciclos** en grafos funcionales (por ejemplo, sucesiones iteradas).
* **Algoritmo de Pollard (rho)** para factorización de enteros.
* **Verificación de terminación** en autómatas y máquinas de estado.


In [9]:
# ============================================================
# Celda 13 — Actividad 5: Algoritmo de Floyd
#              detectar_ciclo(): detecta Y localiza el inicio
# ============================================================

def detectar_ciclo(cabeza: Optional[Nodo]) -> Optional[Nodo]:
    """
    Retorna el nodo donde INICIA el ciclo, o None si no hay ciclo.

    Implementa el algoritmo de Floyd (tortuga y liebre) en dos fases:
      Fase 1: detectar si existe un ciclo.
      Fase 2: localizar el nodo de inicio del ciclo.

    Complejidad: O(n) tiempo, O(1) espacio.
    """
    if cabeza is None:
        return None

    lento: Optional[Nodo] = cabeza
    rapido: Optional[Nodo] = cabeza

    # --------------------------------------------------------
    # Fase 1: detectar si existe ciclo
    # --------------------------------------------------------
    while rapido is not None and rapido.siguiente is not None:
        lento = lento.siguiente              # avanza 1
        rapido = rapido.siguiente.siguiente  # avanza 2
        if lento is rapido:
            break
    else:
        # El while terminó sin break → no hay ciclo
        return None

    # --------------------------------------------------------
    # Fase 2: encontrar el inicio del ciclo
    # --------------------------------------------------------
    lento = cabeza
    while lento is not rapido:
        lento = lento.siguiente
        rapido = rapido.siguiente  # ahora ambos avanzan de a 1
    return lento

In [10]:
# ============================================================
# Pruebas manuales del algoritmo de Floyd
# ============================================================

titulo("Actividad 5 — Pruebas del algoritmo de Floyd")

# ------------------------------------------------------------
# Caso 1: lista con ciclo 1 → 2 → 3 → 4 → 5 → (vuelve a 3)
# ------------------------------------------------------------
nodos = [Nodo(i) for i in range(1, 6)]
for i in range(4):
    nodos[i].siguiente = nodos[i + 1]
nodos[4].siguiente = nodos[2]  # ciclo: 5 apunta a 3

inicio = detectar_ciclo(nodos[0])
if inicio is not None:
    print(f"Ciclo detectado, inicia en nodo con dato = {inicio.dato}")
    assert inicio.dato == 3, "El ciclo debería iniciar en el nodo 3"
    print("Floyd VERIFICADO √")
else:
    print("ERROR: no se detectó el ciclo")

# ------------------------------------------------------------
# Caso 2: lista sin ciclo
# ------------------------------------------------------------
l_sin = ListaEnlazada()
for v in [1, 2, 3, 4, 5]:
    l_sin.insertar_final(v)

resultado = detectar_ciclo(l_sin.cabeza)
assert resultado is None, "No debería haber ciclo"
print("\nSin ciclo: None √")

# ------------------------------------------------------------
# Caso 3: lista vacía
# ------------------------------------------------------------
assert detectar_ciclo(None) is None
print("Lista vacía: None √")

# ------------------------------------------------------------
# Caso 4: un solo nodo sin ciclo
# ------------------------------------------------------------
n = Nodo(42)
assert detectar_ciclo(n) is None
print("Un nodo sin ciclo: None √")

# ------------------------------------------------------------
# Caso 5: un solo nodo con ciclo a sí mismo
# ------------------------------------------------------------
n_self = Nodo(7)
n_self.siguiente = n_self
inicio_self = detectar_ciclo(n_self)
assert inicio_self is n_self
print("Un nodo con auto-ciclo: detectado √")

# ------------------------------------------------------------
# Caso 6: ciclo que empieza en la cabeza
# ------------------------------------------------------------
nodos_c = [Nodo(i) for i in range(1, 4)]
nodos_c[0].siguiente = nodos_c[1]
nodos_c[1].siguiente = nodos_c[2]
nodos_c[2].siguiente = nodos_c[0]  # vuelve a la cabeza
inicio_c = detectar_ciclo(nodos_c[0])
assert inicio_c is nodos_c[0]
print("Ciclo desde la cabeza: detectado en cabeza √")


Actividad 5 — Pruebas del algoritmo de Floyd
Ciclo detectado, inicia en nodo con dato = 3
Floyd VERIFICADO √

Sin ciclo: None √
Lista vacía: None √
Un nodo sin ciclo: None √
Un nodo con auto-ciclo: detectado √
Ciclo desde la cabeza: detectado en cabeza √


In [11]:
# ============================================================
# Celda 14 — Actividad 5: Pruebas unitarias del algoritmo de Floyd
# ============================================================

class TestFloyd(unittest.TestCase):
    """Pruebas unitarias para detectar_ciclo()."""

    # --------------------------------------------------------
    # Utilidad interna: construye una lista a partir de valores
    # y opcionalmente crea un ciclo al nodo en 'pos_ciclo'.
    # --------------------------------------------------------
    @staticmethod
    def _construir(valores, pos_ciclo=None):
        if not valores:
            return None
        nodos = [Nodo(v) for v in valores]
        for i in range(len(nodos) - 1):
            nodos[i].siguiente = nodos[i + 1]
        if pos_ciclo is not None:
            nodos[-1].siguiente = nodos[pos_ciclo]
        return nodos

    # --------------------------------------------------------
    # Casos sin ciclo
    # --------------------------------------------------------
    def test_lista_vacia_sin_ciclo(self) -> None:
        self.assertIsNone(detectar_ciclo(None))

    def test_un_nodo_sin_ciclo(self) -> None:
        n = Nodo(1)
        self.assertIsNone(detectar_ciclo(n))

    def test_lista_sin_ciclo(self) -> None:
        nodos = self._construir([1, 2, 3, 4, 5])
        self.assertIsNone(detectar_ciclo(nodos[0]))

    # --------------------------------------------------------
    # Casos con ciclo
    # --------------------------------------------------------
    def test_ciclo_en_medio(self) -> None:
        # 1 → 2 → 3 → 4 → 5 → (vuelve a 3)
        nodos = self._construir([1, 2, 3, 4, 5], pos_ciclo=2)
        inicio = detectar_ciclo(nodos[0])
        self.assertIsNotNone(inicio)
        self.assertEqual(inicio.dato, 3)

    def test_ciclo_en_cabeza(self) -> None:
        # 1 → 2 → 3 → (vuelve a 1)
        nodos = self._construir([1, 2, 3], pos_ciclo=0)
        inicio = detectar_ciclo(nodos[0])
        self.assertIs(inicio, nodos[0])

    def test_ciclo_en_cola(self) -> None:
        # 1 → 2 → 3 → (vuelve a 3, auto-ciclo en el último)
        nodos = self._construir([1, 2, 3], pos_ciclo=2)
        inicio = detectar_ciclo(nodos[0])
        self.assertIs(inicio, nodos[2])

    def test_un_nodo_con_autociclo(self) -> None:
        n = Nodo(7)
        n.siguiente = n
        self.assertIs(detectar_ciclo(n), n)

    def test_dos_nodos_con_ciclo_mutuo(self) -> None:
        n1 = Nodo(1)
        n2 = Nodo(2)
        n1.siguiente = n2
        n2.siguiente = n1  # ciclo entre ambos
        self.assertIs(detectar_ciclo(n1), n1)

    def test_ciclo_largo(self) -> None:
        # 1 → 2 → ... → 10 → (vuelve a 5)
        nodos = self._construir(list(range(1, 11)), pos_ciclo=4)
        inicio = detectar_ciclo(nodos[0])
        self.assertIsNotNone(inicio)
        self.assertEqual(inicio.dato, 5)

    # --------------------------------------------------------
    # Verificación de que la lista NO se modifica
    # --------------------------------------------------------
    def test_no_modifica_lista_sin_ciclo(self) -> None:
        nodos = self._construir([1, 2, 3, 4, 5])
        datos_antes = [n.dato for n in nodos]
        enlaces_antes = [n.siguiente for n in nodos]
        detectar_ciclo(nodos[0])
        datos_despues = [n.dato for n in nodos]
        enlaces_despues = [n.siguiente for n in nodos]
        self.assertEqual(datos_antes, datos_despues)
        self.assertEqual(enlaces_antes, enlaces_despues)


# ------------------------------------------------------------
# Ejecución de las pruebas en Jupyter
# ------------------------------------------------------------
def ejecutar_pruebas_floyd() -> unittest.result.TestResult:
    loader = unittest.TestLoader()
    suite = loader.loadTestsFromTestCase(TestFloyd)
    runner = unittest.TextTestRunner(verbosity=2)
    return runner.run(suite)


titulo("Actividad 5 — Pruebas unitarias de Floyd")
resultado_floyd = ejecutar_pruebas_floyd()
print(f"\nPruebas ejecutadas: {resultado_floyd.testsRun}")
print(f"Fallos: {len(resultado_floyd.failures)} | Errores: {len(resultado_floyd.errors)}")

test_ciclo_en_cabeza (__main__.TestFloyd.test_ciclo_en_cabeza) ... ok
test_ciclo_en_cola (__main__.TestFloyd.test_ciclo_en_cola) ... ok
test_ciclo_en_medio (__main__.TestFloyd.test_ciclo_en_medio) ... ok
test_ciclo_largo (__main__.TestFloyd.test_ciclo_largo) ... ok
test_dos_nodos_con_ciclo_mutuo (__main__.TestFloyd.test_dos_nodos_con_ciclo_mutuo) ... ok
test_lista_sin_ciclo (__main__.TestFloyd.test_lista_sin_ciclo) ... ok
test_lista_vacia_sin_ciclo (__main__.TestFloyd.test_lista_vacia_sin_ciclo) ... ok
test_no_modifica_lista_sin_ciclo (__main__.TestFloyd.test_no_modifica_lista_sin_ciclo) ... ok
test_un_nodo_con_autociclo (__main__.TestFloyd.test_un_nodo_con_autociclo) ... ok
test_un_nodo_sin_ciclo (__main__.TestFloyd.test_un_nodo_sin_ciclo) ... ok

----------------------------------------------------------------------
Ran 10 tests in 0.045s

OK



Actividad 5 — Pruebas unitarias de Floyd

Pruebas ejecutadas: 10
Fallos: 0 | Errores: 0


## Cierre — Resumen del notebook

En este notebook se implementaron desde cero, en **Python 3.11+**, las estructuras solicitadas en la guía de práctica SIS210 — Semana 5:

### Actividades completadas

| # | Actividad | Implementación | Pruebas |
|---|-----------|----------------|---------|
| 1 | Lista enlazada simple | `Nodo` + `ListaEnlazada` con `cabeza`, `cola` y todas las operaciones | Demostración manual + casos borde |
| 2 | Pruebas unitarias | `TestListaEnlazada` con 12 pruebas | `unittest.TextTestRunner(verbosity=2)` |
| 4 | Lista doblemente enlazada | `NodoDoble` + `ListaDoble` con eliminación O(1) | `TestListaDoble` con 6 pruebas + LRU Cache |
| 5 | Algoritmo de Floyd | `detectar_ciclo()` en dos fases | 6 casos manuales + `TestFloyd` con 10 pruebas |

### Tabla de complejidades implementadas

| Operación | Estructura | Complejidad |
|-----------|------------|-------------|
| Insertar al inicio | Lista simple | **O(1)** |
| Insertar al final | Lista simple (con `cola`) | **O(1)** |
| Insertar en posición k | Lista simple | O(k) |
| Eliminar por dato | Lista simple | O(n) |
| Buscar | Lista simple | O(n) |
| Invertir | Lista simple | O(n) |
| Eliminar nodo dado | Lista doble | **O(1)** |
| Mover al final (LRU) | Lista doble | **O(1)** |
| Detectar ciclo | Floyd | **O(n) tiempo / O(1) espacio** |
| Localizar inicio del ciclo | Floyd (fase 2) | O(n) tiempo / O(1) espacio |

### Conceptos clave reforzados

- **Puntero `cola`**: convierte `insertar_final` de O(n) a O(1) en la lista simple.
- **Doble enlace**: permite eliminar un nodo con solo su referencia, sin buscar el anterior.
- **LRU Cache**: caso real donde la lista doble es la estructura ideal (eliminar y reinsertar en O(1)).
- **Algoritmo de Floyd**: técnica de dos punteros que detecta y localiza ciclos sin memoria adicional.
- **Pruebas unitarias**: cada operación crítica queda verificada de forma automática e independiente.

### Pendientes (fuera del alcance de este notebook)

- **C++17**: implementación de `ListaEnlazada<T>` con destructor y pruebas con `assert` → carpeta `cpp/`.
- **Benchmark** lista vs. vector → `cpp/src/benchmark_lista_vector.cpp`.
- **Investigación Skip List** → `docs/investigacion_skip_list.md`.
- **Preguntas de análisis** → `docs/analisis_preguntas.md`.

---

_Fin del notebook — Práctica N.º 05, SIS210._